# Gemma 4 × Vercel Web Chat UI (Step 25)

Deploy a web chat interface for your local Gemma 4 agent so you can access it
from your phone browser without installing anything on mobile.

Architecture:
- **Backend**: FastAPI server on your desktop, wrapping the LlamaIndex Ollama agent
- **Frontend**: Next.js app deployed to Vercel, proxying to your desktop via a
  secure tunnel (ngrok or Cloudflare Tunnel)
- **Local model**: `gemma4:12b` on Ollama — responses generated on your machine

## What you need
- [Ollama](https://ollama.com) with `gemma4:12b`
- `fastapi`, `uvicorn`, `llama-index-llms-ollama` installed
- Node.js ≥18 (for local Next.js dev)
- [Vercel account](https://vercel.com) (free tier works)
- `ngrok` or Cloudflare Tunnel for exposing your local backend

```bash
pip install fastapi uvicorn llama-index-llms-ollama llama-index-core
```

## Part 1 — FastAPI Backend

Run this on your desktop. It exposes `/chat` and `/stream` endpoints that the
Vercel frontend calls.

In [ ]:
# Save this as gemma4_server.py and run: uvicorn gemma4_server:app --host 0.0.0.0 --port 8080

FASTAPI_SERVER_CODE = '''
from __future__ import annotations

import asyncio
import json
import os
from typing import AsyncGenerator

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

from llama_index.llms.ollama import Ollama
from llama_index.core.llms import ChatMessage

app = FastAPI(title="Gemma 4 Local Chat API")

# Allow Vercel frontend + localhost dev
ALLOWED_ORIGINS = os.environ.get("ALLOWED_ORIGINS", "http://localhost:3000").split(",")
app.add_middleware(
    CORSMiddleware,
    allow_origins=ALLOWED_ORIGINS,
    allow_methods=["POST", "GET"],
    allow_headers=["*"],
)

llm = Ollama(model="gemma4:12b", request_timeout=180.0)


class ChatRequest(BaseModel):
    messages: list[dict]  # [{"role": "user"|"assistant", "content": str}]
    stream: bool = False


@app.get("/health")
async def health() -> dict:
    return {"status": "ok", "model": "gemma4:12b"}


@app.post("/chat")
async def chat(req: ChatRequest) -> dict:
    messages = [ChatMessage(role=m["role"], content=m["content"]) for m in req.messages]
    response = await llm.achat(messages)
    return {"content": str(response.message.content)}


async def _stream_tokens(messages: list[ChatMessage]) -> AsyncGenerator[str, None]:
    async for chunk in await llm.astream_chat(messages):
        if chunk.delta:
            yield f"data: {json.dumps({'delta': chunk.delta})}\\n\\n"
    yield "data: [DONE]\\n\\n"


@app.post("/stream")
async def stream_chat(req: ChatRequest) -> StreamingResponse:
    messages = [ChatMessage(role=m["role"], content=m["content"]) for m in req.messages]
    return StreamingResponse(
        _stream_tokens(messages),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
    )
'''

with open("gemma4_server.py", "w") as f:
    f.write(FASTAPI_SERVER_CODE.strip())

print("gemma4_server.py written.")
print("Start it with: uvicorn gemma4_server:app --host 0.0.0.0 --port 8080")

## Part 2 — Tunnel Setup

Expose your local port 8080 so Vercel can reach it. Two options:

In [ ]:
# Option A: ngrok (easiest)
print("""Option A — ngrok:
  1. Install: https://ngrok.com/download
  2. Run: ngrok http 8080
  3. Copy the https://xxxx.ngrok.io URL — that's your BACKEND_URL for Vercel

Option B — Cloudflare Tunnel (stable, free, no time limit):
  1. Install cloudflared: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps/install-and-setup/
  2. Run: cloudflared tunnel --url http://localhost:8080
  3. Use the *.trycloudflare.com URL as your BACKEND_URL

Once you have the URL, set it as an environment variable in Vercel:
  NEXT_PUBLIC_BACKEND_URL=https://your-tunnel-url.ngrok.io
""")

## Part 3 — Next.js Frontend

A minimal chat UI that connects to your local backend.

In [ ]:
import os
from pathlib import Path

# Create the Next.js app structure
NEXTJS_DIR = Path("gemma4-chat-ui")
NEXTJS_DIR.mkdir(exist_ok=True)
(NEXTJS_DIR / "app").mkdir(exist_ok=True)
(NEXTJS_DIR / "components").mkdir(exist_ok=True)

# package.json
(NEXTJS_DIR / "package.json").write_text(json.dumps({
    "name": "gemma4-chat-ui",
    "version": "0.1.0",
    "private": True,
    "scripts": {"dev": "next dev", "build": "next build", "start": "next start"},
    "dependencies": {"next": "15.0.0", "react": "^19", "react-dom": "^19"},
    "devDependencies": {"typescript": "^5", "@types/react": "^19", "@types/node": "^22"}
}, indent=2))

# next.config.js
(NEXTJS_DIR / "next.config.js").write_text("""
/** @type {import('next').NextConfig} */
const nextConfig = {};
module.exports = nextConfig;
""".strip())

print(f"Next.js project scaffold created at ./{NEXTJS_DIR}")

In [ ]:
import json

# Main chat page — app/page.tsx
PAGE_TSX = '''
"use client";
import { useState, useRef, useEffect } from "react";

const BACKEND = process.env.NEXT_PUBLIC_BACKEND_URL || "http://localhost:8080";

type Message = { role: "user" | "assistant"; content: string };

export default function Home() {
  const [messages, setMessages] = useState<Message[]>([]);
  const [input, setInput] = useState("");
  const [loading, setLoading] = useState(false);
  const bottomRef = useRef<HTMLDivElement>(null);

  useEffect(() => { bottomRef.current?.scrollIntoView({ behavior: "smooth" }); }, [messages]);

  async function send() {
    if (!input.trim() || loading) return;
    const userMsg: Message = { role: "user", content: input };
    const history = [...messages, userMsg];
    setMessages(history);
    setInput("");
    setLoading(true);

    try {
      // Streaming chat
      const res = await fetch(`${BACKEND}/stream`, {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ messages: history }),
      });

      const reader = res.body!.getReader();
      const decoder = new TextDecoder();
      let assistantContent = "";
      setMessages([...history, { role: "assistant", content: "" }]);

      while (true) {
        const { done, value } = await reader.read();
        if (done) break;
        const text = decoder.decode(value);
        for (const line of text.split("\\n")) {
          if (!line.startsWith("data: ")) continue;
          const payload = line.slice(6);
          if (payload === "[DONE]") break;
          try {
            const { delta } = JSON.parse(payload);
            assistantContent += delta;
            setMessages([...history, { role: "assistant", content: assistantContent }]);
          } catch {}
        }
      }
    } catch (e) {
      setMessages([...history, { role: "assistant", content: `Error: ${e}` }]);
    } finally {
      setLoading(false);
    }
  }

  return (
    <main style={{ maxWidth: 700, margin: "0 auto", padding: 16, fontFamily: "system-ui" }}>
      <h1 style={{ fontSize: 20, marginBottom: 8 }}>Gemma 4 — Local Chat</h1>
      <div style={{ height: "70vh", overflowY: "auto", border: "1px solid #ddd",
                    borderRadius: 8, padding: 12, marginBottom: 12 }}>
        {messages.map((m, i) => (
          <div key={i} style={{
            marginBottom: 12,
            textAlign: m.role === "user" ? "right" : "left",
          }}>
            <span style={{
              display: "inline-block", maxWidth: "80%", padding: "8px 12px",
              borderRadius: 12, whiteSpace: "pre-wrap",
              background: m.role === "user" ? "#0070f3" : "#f0f0f0",
              color: m.role === "user" ? "white" : "black",
            }}>{m.content}</span>
          </div>
        ))}
        {loading && <div style={{ color: "#888", fontSize: 13 }}>Gemma is thinking...</div>}
        <div ref={bottomRef} />
      </div>
      <div style={{ display: "flex", gap: 8 }}>
        <input
          value={input}
          onChange={e => setInput(e.target.value)}
          onKeyDown={e => e.key === "Enter" && !e.shiftKey && send()}
          placeholder="Message Gemma 4..."
          style={{ flex: 1, padding: "10px 14px", borderRadius: 8,
                   border: "1px solid #ccc", fontSize: 15 }}
        />
        <button onClick={send} disabled={loading}
          style={{ padding: "10px 20px", borderRadius: 8, background: "#0070f3",
                   color: "white", border: "none", cursor: "pointer", fontSize: 15 }}>
          {loading ? "..." : "Send"}
        </button>
      </div>
    </main>
  );
}
'''

(NEXTJS_DIR / "app" / "page.tsx").write_text(PAGE_TSX.strip())

# app/layout.tsx
(NEXTJS_DIR / "app" / "layout.tsx").write_text("""
export default function RootLayout({ children }: { children: React.ReactNode }) {
  return <html lang="en"><body>{children}</body></html>;
}
""".strip())

print("Next.js UI files written to gemma4-chat-ui/")

## Part 4 — Deploy to Vercel

In [ ]:
print("""Deploy steps:

1. Push gemma4-chat-ui/ to a new GitHub repo (or a subfolder):
     cd gemma4-chat-ui && git init && git add . && git commit -m 'init'
     gh repo create gemma4-chat-ui --public --source=. --push

2. Import the repo on Vercel:
     vercel.com/new → import from GitHub → select gemma4-chat-ui

3. Add environment variable in Vercel project settings:
     NEXT_PUBLIC_BACKEND_URL = https://your-ngrok-or-cloudflare-url

4. Deploy — Vercel builds and hosts the Next.js frontend automatically.

5. Open the Vercel URL on your phone — you now have a mobile chat UI
   that talks to your local Gemma 4 model!

To redeploy after changes:
     vercel deploy --prod
""")

## Part 5 — Local dev test

In [ ]:
import httpx
import asyncio

async def test_backend(base_url: str = "http://localhost:8080") -> None:
    async with httpx.AsyncClient(timeout=60) as client:
        # Health check
        try:
            health = await client.get(f"{base_url}/health")
            print("Health:", health.json())
        except Exception as e:
            print(f"Backend not reachable at {base_url} — start gemma4_server.py first. ({e})")
            return

        # Chat
        resp = await client.post(f"{base_url}/chat", json={
            "messages": [{"role": "user", "content": "Say hello in 10 words."}]
        })
        print("Chat response:", resp.json()["content"])

await test_backend()